
# 더 빠른 점진법 시험: 기존 `tuning.py` vs 빠른 방식 (EPL 실제 데이터)

**질문:** 남들이 쓰는 점진법(하나 바꾸고 → 결과를 기다려 → 판정)보다 **더 빨리, 그러면서도 덜 틀리게** 좋은 규칙을 찾을 수 있을까?

**비교하는 방식**

| 방식 | 판정 시점 | 판정 기준 |
|---|---|---|
| 흔한 방식 | 시즌 끝 | 평균이 조금이라도 좋으면 채택 |
| **기존 점진법** (`llm-reliability/llmrel/tuning.py`의 `certain`) | 시즌 끝 | 95% 구간 하한이 가장 큰 후보, 하한 > 0이면 채택 |
| 기존 점진법 (4시즌 모아 한 번에) | 4시즌 뒤 | 위와 같음. 지금 `tuning_demo.py`가 쓰는 방식 |
| 빠른-순차 | 매 라운드(10경기) | 언제 멈춰도 유효한 신뢰 시퀀스, 후보 수로 나눈 엄격한 기준 |
| **빠른-순차+스코어** | 매 라운드 | 위 판정을 **정확한 스코어의 로그손실**로 함 (승·무·패보다 정보가 많음) |
| 빠른-순차+스코어+안전장치 | 매 라운드 | 위 + 결과(RPS)가 나빠지는 중이면 보류 |

**두 단계로 잽니다**
- **A. 정답을 아는 시뮬레이션:** 후보 13개(시작 규칙의 이웃)의 실제 효과와 잡음을 EPL에서 재고, 그 값으로 500번 경주시킵니다. 맞는 후보를 고르는 비율, 해로운 후보를 고르는 비율, 걸린 경기 수를 잽니다.
- **B. 실제 EPL 재생:** 2018-19 ~ 2021-22 시즌(1,520경기)을 실시간처럼 흘려보내며 각 방식이 규칙을 바꾸게 합니다. 결과는 **따로 떼어 둔 2022-23 ~ 2026-27 시즌(1,570경기)**으로 채점합니다.

모델은 `llmrel/epl.py`의 온라인 포아송(골 모드)과 같고, 후보 격자도 같습니다. xG(`g`)는 공개 재배포 문제로 뺐습니다. 데이터는 openfootball(CC0)입니다.


In [ ]:
import json, os, urllib.request
from math import exp, factorial
import numpy as np, pandas as pd

SEASONS = [f"{y}-{str(y + 1)[2:]}" for y in range(2010, 2027)]
EPL_URLS = ["https://raw.githubusercontent.com/quanter02/chelsea-analytics/claude/great-dirac-3wwgzy/experiments/data_epl/en1_{s}.json",
            "https://raw.githubusercontent.com/openfootball/football.json/master/{s}/en.1.json"]

def _norm(t):
    t = t.strip()
    for suf in (" AFC", " FC"):
        if t.endswith(suf): t = t[:-len(suf)]
    return t[4:] if t.startswith("AFC ") else t

def load_epl(local="data_epl"):
    rows = []
    for s in SEASONS:
        f = os.path.join(local, f"en1_{s}.json")
        if os.path.exists(f):
            ms = json.load(open(f, encoding="utf-8"))["matches"]
        else:
            for u in EPL_URLS:
                try: ms = json.loads(urllib.request.urlopen(u.format(s=s), timeout=30).read())["matches"]; break
                except Exception: ms = None
        for x in ms:
            sc = x.get("score"); ft = sc.get("ft") if isinstance(sc, dict) else (sc if isinstance(sc, list) and len(sc) == 2 else None)
            if not ft: continue
            date = x["date"]
            if s == "2019-20" and date < "2019-08-01": date = "2020" + date[4:]   # 원본 날짜 오류 수정
            rows.append(dict(season=s, date=date, home=_norm(x["team1"]), away=_norm(x["team2"]), hg=ft[0], ag=ft[1]))
    df = pd.DataFrame(rows).sort_values("date", kind="stable").reset_index(drop=True)
    df["y"] = np.select([df.hg > df.ag, df.hg == df.ag], [0, 1], 2)
    return df

# llm-reliability/llmrel/epl.py 의 시작 규칙과 후보 격자 (xG 관련 g 제외)
START = dict(k=0.05, h=0.25, c=0.8, p=-0.2, rho=0.0, w=0.0, cap=99.0, e=1.0)
GRID = {"k": [0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.1], "h": [0.1, 0.15, 0.2, 0.25, 0.3, 0.35],
        "c": [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0], "p": [0.0, -0.1, -0.2, -0.3, -0.4], "rho": [0.0, 0.05, 0.1, 0.15, 0.2, 0.25],
        "w": [0.0, 0.05, 0.1, 0.15, 0.2, 0.3], "cap": [99.0, 4.0, 3.0, 2.5, 2.0], "e": [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]}
BASE = np.array([0.46, 0.25, 0.29])
MAXG = 10
_G = np.arange(MAXG + 1); _F = np.array([factorial(i) for i in range(MAXG + 1)], float)

def run_model(df, k, h, c, p, rho, w, cap, e, early=6, mu0=0.3):
    """epl.run과 같은 온라인 포아송 (골 모드). 경기별 RPS(결과)와 스코어 로그손실(정확한 스코어)을 함께 반환."""
    a, d, played_n = {}, {}, {}
    mu = mu0; season = None
    rps = np.zeros(len(df)); nll = np.zeros(len(df))
    teams_by = {s: set(g.home) | set(g.away) for s, g in df.groupby("season")}
    for i, r in enumerate(df.itertuples()):
        if r.season != season:
            for t in list(a): a[t] *= c; d[t] *= c
            for t in teams_by[r.season] - set(a): a[t], d[t] = p, p
            season = r.season; played_n = {}
        lh = exp(mu + h + a[r.home] - d[r.away]); la = exp(mu + a[r.away] - d[r.home])
        ph = np.exp(-lh) * lh ** _G / _F; pa = np.exp(-la) * la ** _G / _F
        m = np.outer(ph, pa); m[np.diag_indices_from(m)] *= (1 + rho); m /= m.sum()
        P = (1 - w) * np.array([np.tril(m, -1).sum(), np.trace(m), np.triu(m, 1).sum()]) + w * BASE
        o = np.zeros(3); o[r.y] = 1
        rps[i] = ((np.cumsum(P)[:2] - np.cumsum(o)[:2]) ** 2).sum() / 2
        nll[i] = -np.log(max(m[min(r.hg, MAXG), min(r.ag, MAXG)], 1e-12))     # 스코어 확률 (w 보정 전 분포)
        th, ta = min(r.hg, cap), min(r.ag, cap)
        eh, ea = th - lh, ta - la
        kh = k * (e if played_n.get(r.home, 0) < early else 1.0); ka = k * (e if played_n.get(r.away, 0) < early else 1.0)
        a[r.home] += kh * eh; d[r.away] -= ka * eh; a[r.away] += ka * ea; d[r.home] -= kh * ea
        played_n[r.home] = played_n.get(r.home, 0) + 1; played_n[r.away] = played_n.get(r.away, 0) + 1
        mu += k * 0.05 * (eh + ea)
    return rps, nll


In [ ]:
from math import log, sqrt

ALPHA = 0.05      # 전체 오채택 허용 (후보 수로 나눠 씀)
SEASON = 380      # 기존 방식의 판정 단위: 한 시즌
CHECK = 10        # 빠른 방식의 판정 주기: 한 라운드(10경기)
MIN_N = 30        # 분산 추정을 위한 최소 경기 수
N0 = 200          # 신뢰 시퀀스가 가장 좁아지게 맞추는 경기 수

def cs_lower(m, s, n, alpha, n0=N0):
    """언제 멈춰도 유효한 신뢰 시퀀스(정규 혼합)의 하한. m=평균, s=표준편차, n=표본 수."""
    rho2 = (-2 * log(alpha) + log(-2 * log(alpha) + 1)) / n0
    return m - s * sqrt(2 * (n * rho2 + 1) / (n * n * rho2) * log(sqrt(n * rho2 + 1) / alpha))

def fixed_lower(m, s, n):
    """고정 표본 95% 하한 (tuning.py의 부트스트랩 95% 구간을 정규 근사)."""
    return m - 1.96 * s / sqrt(n)

def decide(D, P, n, method, K):
    """경기 n개가 쌓인 시점의 판정. D, P: 후보별 (합, 제곱합) — D는 결과(RPS) 차이, P는 스코어 로그손실 차이.
    차이 = 현재 규칙 손실 − 후보 손실 (양수면 후보가 좋음). 채택할 후보 번호 또는 None."""
    def stats(S):
        out = []
        for s1, s2 in S:
            m = s1 / n; v = max(s2 / n - m * m, 1e-18) * n / (n - 1)
            out.append((m, sqrt(v)))
        return out
    if method in ("흔한 방식", "기존 점진법"):
        if n % SEASON: return None
        st = stats(D)
        if method == "흔한 방식":
            j = max(range(K), key=lambda i: st[i][0]); return j if st[j][0] > 0 else None
        lows = [fixed_lower(m, s, n) for m, s in st]
        j = max(range(K), key=lambda i: lows[i]); return j if lows[j] > 0 else None
    if n < MIN_N or n % CHECK: return None
    a = ALPHA / K
    if method == "빠른-순차":
        st = stats(D)
        lows = [cs_lower(m, s, n, a) for m, s in st]
        j = max(range(K), key=lambda i: lows[i]); return j if lows[j] > 0 else None
    if method in ("빠른-순차+스코어", "빠른-순차+스코어+안전장치"):
        sp, sd = stats(P), stats(D)
        lows = [cs_lower(m, s, n, a) if s > 1e-12 else -1 for m, s in sp]
        j = max(range(K), key=lambda i: lows[i])
        if lows[j] <= 0: return None
        if method.endswith("안전장치") and sd[j][0] < 0: return None   # 스코어로 빨리 판정하되, 결과(RPS)가 나빠지는 중이면 보류
        return j
    raise ValueError(method)

METHODS = ["흔한 방식", "기존 점진법", "빠른-순차", "빠른-순차+스코어", "빠른-순차+스코어+안전장치"]


In [ ]:
DECIDE = ("2018-08-01", "2022-08-01")   # 실시간처럼 흘려보내며 규칙을 바꾸는 기간 (4시즌)
TRUTH = ("2022-08-01", "2099-01-01")    # 채점에만 쓰는 기간 (2022-23 ~ 2026-27 시즌 초)

_cache = {}
def losses(params):
    key = tuple(sorted(params.items()))
    if key not in _cache:
        _cache[key] = run_model(df, **params)
    return _cache[key]

def neighbors(cur):
    out = []
    for p, vals in GRID.items():
        i = vals.index(cur[p])
        for j in (i - 1, i + 1):
            if 0 <= j < len(vals): out.append((f"{p}={vals[j]}", {**cur, p: vals[j]}))
    return out

def candidate_stats():
    """시작 규칙의 이웃 후보별: 결정 기간·진실 기간의 효과(×1000)와 경기별 잡음. 시뮬레이션의 '정답'으로 씀."""
    dec = ((df.date >= DECIDE[0]) & (df.date < DECIDE[1])).to_numpy(); tru = ((df.date >= TRUTH[0]) & (df.date < TRUTH[1])).to_numpy()
    r0, n0 = losses(START)
    rows = []
    for name, c in neighbors(START):
        r, n = losses(c); dr, dn = r0 - r, n0 - n
        rows.append(dict(후보=name, rps_dec=dr[dec].mean() * 1000, rps_tru=dr[tru].mean() * 1000,
                         rps_tru_t=dr[tru].mean() / dr[tru].std() * np.sqrt(tru.sum()),
                         nll_dec=dn[dec].mean() * 1000, nll_eff=dn[tru].mean() * 1000,
                         sd_rps=dr[dec].std() * 1000, sd_nll=dn[dec].std() * 1000,
                         corr=np.corrcoef(dr[dec], dn[dec])[0, 1] if dn[dec].std() > 0 else 0.0))
    return pd.DataFrame(rows)


df = load_epl()
print(f"경기 {len(df):,}개 ({df.date.min()} ~ {df.date.max()})")
S = candidate_stats()
S.round(2)


**후보 13개의 실제 효과 (×1000, 양수면 시작 규칙보다 좋음)**
- `rps_tru`: 진실 기간 결과(RPS) 효과. **진짜로 좋은 후보는 `k=0.04` 하나뿐**입니다(t ≈ 3.3).
- `rps_dec`: 결정 기간 효과. `c=0.9`(+0.54)와 `h=0.2`(+0.47)는 결정 기간에 좋아 보이지만 진실 기간 효과는 0입니다. **함정 후보**입니다.
- `nll_eff`: 스코어 로그손실 효과. 잡음 대비 신호가 결과(RPS)의 약 2배입니다(`k=0.04`). 다만 `cap=4.0`처럼 스코어로는 좋아 보이는데 결과로는 효과가 작은 후보도 있습니다.


## A. 정답을 아는 시뮬레이션 (500회)

In [ ]:
import numpy as np, pandas as pd

def simulate(stats, method, T=1520, reps=500, seed=0, good_min=0.5):
    """정답을 아는 경주. stats: 후보별 실제 효과(×1000)와 경기별 잡음(EPL에서 잰 값).
    첫 채택이 진짜 좋은 후보(효과 ≥ good_min)인지, 해로운 후보(효과 < 0)인지, 몇 경기 만인지."""
    rng = np.random.default_rng(seed)
    mu_r, mu_n = stats.rps_tru.to_numpy() / 1000, stats.nll_eff.to_numpy() / 1000
    sd_r, sd_n, rho = stats.sd_rps.to_numpy() / 1000, stats.sd_nll.to_numpy() / 1000, stats["corr"].fillna(0).to_numpy()
    K = len(stats); good = stats.rps_tru.to_numpy() >= good_min; bad = stats.rps_tru.to_numpy() < 0
    res = []
    for _ in range(reps):
        z1 = rng.standard_normal((T, K)); z2 = rho * z1 + np.sqrt(1 - rho ** 2) * rng.standard_normal((T, K))
        dr = mu_r + sd_r * z1; dn = mu_n + sd_n * z2
        cr, cr2, cn, cn2 = (np.cumsum(x, 0) for x in (dr, dr ** 2, dn, dn ** 2))
        pick, when = None, None
        for n in range(1, T + 1):
            D = list(zip(cr[n - 1], cr2[n - 1])); P = list(zip(cn[n - 1], cn2[n - 1]))
            j = decide(D, P, n, method, K)
            if j is not None: pick, when = j, n; break
        res.append(dict(pick=pick, when=when, good=pick is not None and good[pick], bad=pick is not None and bad[pick]))
    R = pd.DataFrame(res)
    g = R[R.good]
    return dict(방식=method, 맞는_채택=round(R.good.mean(), 3), 해로운_채택=round(R.bad.mean(), 3),
                채택_없음=round(R.pick.isna().mean(), 3), 맞는_채택까지_경기_중앙값=float(g.when.median()) if len(g) else np.nan,
                맞는_채택까지_시즌=round(float(g.when.median()) / 380, 2) if len(g) else np.nan)


sim = pd.DataFrame([simulate(S, m, reps=500) for m in METHODS]).set_index("방식")
sim

## B. 실제 EPL 재생 (결정 2018-19 ~ 2021-22, 채점 2022-23 ~ 2026-27)

In [ ]:
def replay(method):
    """결정 기간(2018-19 ~ 2021-22)을 경기 순서대로 흘려보내며 규칙을 바꾼다. 채택하면 그 뒤 경기부터 새로 센다(실시간과 같은 조건)."""
    idx = np.flatnonzero(((df.date >= DECIDE[0]) & (df.date < DECIDE[1])).to_numpy())
    cur, start, log_ = dict(START), 0, []
    while start < len(idx):
        cands = neighbors(cur); K = len(cands)
        r0, n0 = losses(cur)
        seg = idx[start:]
        dr = np.array([r0[seg] - losses(c)[0][seg] for _, c in cands]).T
        dn = np.array([n0[seg] - losses(c)[1][seg] for _, c in cands]).T
        cr, cr2, cn, cn2 = (np.cumsum(x, 0) for x in (dr, dr ** 2, dn, dn ** 2))
        pick = None
        for n in range(1, len(seg) + 1):
            # 기존 방식의 판정 시점은 '시즌 끝'(누적 경기 수가 아니라 실제 시즌 경계)에 맞춤
            if method in ("흔한 방식", "기존 점진법"):
                last = seg[n - 1]
                if n < len(seg) and df.season.iat[seg[n]] == df.season.iat[last]: continue
                st = [(cr[n - 1, i] / n, np.sqrt(max(cr2[n - 1, i] / n - (cr[n - 1, i] / n) ** 2, 1e-18) * n / (n - 1))) for i in range(K)]
                if method == "흔한 방식":
                    j = max(range(K), key=lambda i: st[i][0]); j = j if st[j][0] > 0 else None
                else:
                    lows = [fixed_lower(m, s, n) for m, s in st]; j = max(range(K), key=lambda i: lows[i]); j = j if lows[j] > 0 else None
            else:
                j = decide(list(zip(cr[n - 1], cr2[n - 1])), list(zip(cn[n - 1], cn2[n - 1])), n, method, K)
            if j is not None:
                pick = (n, j); break
        if pick is None: break
        n, j = pick
        log_.append(dict(방식=method, 날짜=df.date.iat[seg[n - 1]], 시즌=df.season.iat[seg[n - 1]], 판정까지_경기=n, 바꾼규칙=cands[j][0]))
        cur, start = cands[j][1], start + n
    return cur, pd.DataFrame(log_)

def offline_hill_climb():
    """기존 점진법 원형(tuning.py): 결정 기간 4시즌을 다 모은 뒤 한 번에 '가장 확실한 개선'을 반복 채택 (부트스트랩 95%)."""
    msk = ((df.date >= DECIDE[0]) & (df.date < DECIDE[1])).to_numpy()
    rng = np.random.default_rng(0)
    cur = dict(START); log_ = []
    while True:
        r0 = losses(cur)[0][msk]; best = None
        for name, c in neighbors(cur):
            d = r0 - losses(c)[0][msk]
            boots = d[rng.integers(0, len(d), (1000, len(d)))].mean(1)
            lo = np.percentile(boots, 2.5)
            if best is None or lo > best[0]: best = (lo, name, c)
        if best[0] <= 0: break
        log_.append(dict(방식="기존 점진법 (4시즌 모아 한 번에)", 날짜=DECIDE[1], 시즌="2021-22 끝", 판정까지_경기=int(msk.sum()), 바꾼규칙=best[1]))
        cur = best[2]
    return cur, pd.DataFrame(log_)

def truth_gain(params, block=10, reps=2000, seed=0):
    """진실 기간 RPS 개선(시작 규칙 대비, ×1000)과 경기 10개 블록 부트스트랩 95% 구간."""
    tru = ((df.date >= TRUTH[0]) & (df.date < TRUTH[1])).to_numpy()
    d = (losses(START)[0] - losses(params)[0])[tru] * 1000
    nb = len(d) // block; b = d[: nb * block].reshape(nb, block).mean(1)
    m = np.random.default_rng(seed).choice(b, (reps, nb)).mean(1)
    return d.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)

def realized_during(log_):
    """결정 기간 동안 '그때그때 쓰고 있던 규칙'의 RPS 개선(시작 규칙 대비, ×1000). 일찍 채택할수록 일찍 이득."""
    msk = ((df.date >= DECIDE[0]) & (df.date < DECIDE[1])).to_numpy()
    idx = np.flatnonzero(msk); cur = dict(START); base = losses(START)[0]
    changes = {} if log_.empty else {d: r for d, r in zip(log_.날짜, log_.바꾼규칙)}
    out = np.zeros(len(idx))
    for t, i in enumerate(idx):
        out[t] = (base[i] - losses(cur)[0][i]) * 1000
        dte = df.date.iat[i]
        if dte in changes and (t + 1 == len(idx) or df.date.iat[idx[t + 1]] != dte):
            p, v = changes[dte].split("="); cur = {**cur, p: type(START[p])(float(v))}
    return out.mean()


rows, logs = [], []
for m in METHODS:
    cur, L = replay(m); logs.append(L)
    g, lo, hi = truth_gain(cur)
    rows.append(dict(방식=m, 채택수=len(L), 첫채택=L.날짜.iat[0] if len(L) else "-",
                     첫채택까지_경기=int(L.판정까지_경기.iat[0]) if len(L) else None,
                     최종규칙=", ".join(f"{k}={v}" for k, v in cur.items() if v != START[k]) or "(변화 없음)",
                     학습중_실제이득=round(realized_during(L), 2), 진실기간_개선=f"{g:+.2f} [{lo:+.2f}, {hi:+.2f}]"))
cur, L = offline_hill_climb(); logs.append(L); g, lo, hi = truth_gain(cur)
rows.append(dict(방식="기존 점진법 (4시즌 모아 한 번에)", 채택수=len(L), 첫채택="2022-08-01 (4시즌 뒤)", 첫채택까지_경기=1520,
                 최종규칙=", ".join(f"{k}={v}" for k, v in cur.items() if v != START[k]), 학습중_실제이득=0.0,
                 진실기간_개선=f"{g:+.2f} [{lo:+.2f}, {hi:+.2f}]"))
display(pd.DataFrame(rows).set_index("방식"))
pd.concat(logs).reset_index(drop=True)



## 결과와 해석 (사전 실행)

**A. 시뮬레이션 (500회, 정답을 앎)**

| 방식 | 맞는 후보 채택 | 해로운 후보 채택 | 채택 없음 | 맞는 채택까지 |
|---|---|---|---|---|
| 흔한 방식 | 64% | 5% | 0% | 1.0시즌 (시즌 끝) |
| 기존 점진법 | 63% | 5% | 2% | 1.0시즌 (시즌 끝) |
| 빠른-순차 (결과만) | 26% | 0% | **71%** | 2.9시즌 |
| **빠른-순차+스코어** | **99%** | **0%** | 0% | **0.9시즌** |
| 빠른-순차+스코어+안전장치 | 99% | 0% | 0% | 0.9시즌 |

(정확한 숫자는 위 셀 출력. 표는 100회 시험 실행 기준으로 썼고, 500회 결과도 같은 방향입니다.)

**B. 실제 EPL 재생**

| 방식 | 첫 채택 | 채택한 규칙 (순서) | 진실 기간 개선 (95%) |
|---|---|---|---|
| 흔한 방식 | 2019-05 (380경기) | k=0.04 → **c=0.9** → **h=0.2** → k=0.03 | +1.75 [+0.32, +3.20] |
| 기존 점진법 (실시간) | 2020-07 (760경기) | k=0.04 → **h=0.2** | +1.03 [+0.19, +1.85] |
| 기존 점진법 (4시즌 모아 한 번에) | **2022-08 (1,520경기)** | k=0.04 → **c=0.9** → k=0.03 | +1.71 [+0.41, +3.01] |
| 빠른-순차 (결과만) | 없음 | — | 0 |
| **빠른-순차+스코어** | **2019-03 (280경기)** | k=0.04 → k=0.03 | **+1.60 [+0.42, +2.81]** |

굵은 글씨 규칙(`c=0.9`, `h=0.2`)은 진실 기간 효과가 0인 함정 후보입니다.

### 알게 된 것
1. **속도는 판정을 자주 하는 데서 나오지 않습니다.** 매 라운드 판정(빠른-순차)만 하면, 후보가 많을 때 엄격한 기준을 넘지 못해 **오히려 느려집니다**(71%는 4시즌 안에 채택 없음).
2. **속도는 정보량에서 나옵니다.** 같은 경기에서 승·무·패 대신 정확한 스코어까지 쓰면 잡음 대비 신호가 약 2배가 되고, 필요한 경기 수는 약 1/4로 줄어듭니다.
   - 시뮬레이션: 맞는 채택 63% → 99%, 해로운 채택 5% → 0%
   - 실제 재생: 첫 올바른 채택이 기존 실시간 방식보다 **1년 4개월**, 4시즌 모아 한 번에 하는 방식보다 **3년 5개월** 빨랐습니다.
3. **빠르면서도 덜 틀립니다.** 빠른 방식은 함정 후보를 한 번도 채택하지 않았습니다. 기존 방식들은 모두 하나 이상 채택했습니다.
4. **정직한 한계:**
   - 실제 재생 한 번에서는 함정 후보가 해롭지 않고 중립이었습니다. 그래서 최종 진실 기간 개선은 방식들 사이에 **구간이 겹칩니다**(+1.0 ~ +1.75).
   - 학습 기간에 실제로 얻은 이득도 흔한 방식이 1.11로 가장 컸습니다. 결정 기간에는 함정 후보가 우연히 이득이었기 때문입니다.
   - **빠른 방식의 확실한 이점은 "더 일찍, 함정 없이"이고, 최종 성능이 더 높다는 증거는 아직 없습니다.**
5. 이 결론은 오늘 정리한 원칙("계산을 자주 돌리는 것은 한계를 줄이지 못한다. 정보량을 늘리거나 변화를 감지해야 한다")을 실제 데이터로 다시 확인한 것입니다.

### 다른 영역에 옮기는 법
- **"결과"보다 정보가 많은 같은 시점의 신호를 찾습니다:**
  - 축구: 승·무·패 → 스코어(→ xG)
  - 지자체 경보: 연간 확정치 → 월별·연령별 세부 자료
  - LLM 판단: 정오 → 확신도와 근거 검증 결과
- 그 신호로 **빠르게 판정**하고, 결과로는 **해로운지만 확인**합니다(안전장치).
- 채택 기준은 언제 멈춰도 유효한 신뢰 시퀀스에 후보 수 보정을 합니다. 많이 시도해도 오채택이 늘지 않습니다.



## C. 다른 기간으로 재현

B는 기간 하나(2018~2022 결정)뿐이었습니다. 같은 실험을 다른 기간에 반복합니다. **결과를 보기 전에 정한 것:**

| 기간 | 결정 (실시간 재생) | 채점 |
|---|---|---|
| W1 | 2011-12 ~ 2014-15 | 2015-16 ~ 2018-19 |
| W2 | 2015-16 ~ 2018-19 | 2019-20 ~ 2022-23 |
| W3 | 2018-19 ~ 2021-22 | 2022-23 ~ 2026-27 (B와 같음) |

- **후보 분류 (채점 기간 기준):** t ≥ 2이면 좋은 후보, 효과 < 0이면 해로운 후보, 그 사이는 중립.
- **재현 성공 조건:**
  1. 세 기간 모두 시뮬레이션에서 빠른 방식(판정 + 스코어 정보)이 기존 점진법보다 좋은 후보를 **더 많이** 고를 것
  2. 해로운 후보를 **더 적게** 고를 것
  3. 실제 재생에서 해로운 후보를 채택하지 않을 것


In [ ]:
WINDOWS = {   # (결정 기간, 채점 기간) — 결과를 보기 전에 고정
    "W1 결정 11-15 → 채점 15-19": (("2011-08-01", "2015-08-01"), ("2015-08-01", "2019-08-01")),
    "W2 결정 15-19 → 채점 19-23": (("2015-08-01", "2019-08-01"), ("2019-08-01", "2023-08-01")),
    "W3 결정 18-22 → 채점 22-27": (("2018-08-01", "2022-08-01"), ("2022-08-01", "2099-01-01")),
}
GOOD_T = 2.0   # 채점 기간 t ≥ 2 → 좋은 후보, 효과 < 0 → 해로운 후보, 나머지는 중립

def simulate_window(stats, method, reps=500, seed=0):
    """simulate와 같지만 좋은/해로운 후보를 채점 기간 t 기준으로 정함."""
    rng = np.random.default_rng(seed)
    mu_r, mu_n = stats.rps_tru.to_numpy() / 1000, stats.nll_eff.to_numpy() / 1000
    sd_r, sd_n, rho = stats.sd_rps.to_numpy() / 1000, stats.sd_nll.to_numpy() / 1000, stats["corr"].fillna(0).to_numpy()
    K = len(stats); T = 1520
    good = (stats.rps_tru_t >= GOOD_T).to_numpy(); bad = (stats.rps_tru < 0).to_numpy()
    res = []
    for _ in range(reps):
        z1 = rng.standard_normal((T, K)); z2 = rho * z1 + np.sqrt(1 - rho ** 2) * rng.standard_normal((T, K))
        dr = mu_r + sd_r * z1; dn = mu_n + sd_n * z2
        cr, cr2, cn, cn2 = (np.cumsum(x, 0) for x in (dr, dr ** 2, dn, dn ** 2))
        pick = when = None
        for n in range(1, T + 1):
            j = decide(list(zip(cr[n - 1], cr2[n - 1])), list(zip(cn[n - 1], cn2[n - 1])), n, method, K)
            if j is not None: pick, when = j, n; break
        res.append(dict(pick=pick, when=when, good=pick is not None and good[pick], bad=pick is not None and bad[pick]))
    R = pd.DataFrame(res); g = R[R.good]
    return dict(방식=method, 좋은후보수=int(good.sum()), 맞는_채택=round(R.good.mean(), 3), 해로운_채택=round(R.bad.mean(), 3),
                채택_없음=round(R.pick.isna().mean(), 3), 맞는_채택까지_시즌=round(float(g.when.median()) / 380, 2) if len(g) else np.nan)

def replicate():
    global DECIDE, TRUTH
    sims, reps_, cands = [], [], []
    for wname, (dec, tru) in WINDOWS.items():
        DECIDE, TRUTH = dec, tru
        S = candidate_stats(); S.insert(0, "기간", wname); cands.append(S)
        good = set(S.후보[S.rps_tru_t >= GOOD_T]); bad = set(S.후보[S.rps_tru < 0])
        for m in ["기존 점진법", "빠른-순차+스코어", "빠른-순차+스코어+안전장치"]:
            r = simulate_window(S, m); r["기간"] = wname; sims.append(r)
        for m in ["흔한 방식", "기존 점진법", "빠른-순차+스코어", "빠른-순차+스코어+안전장치"]:
            cur, L = replay(m)
            # 첫 '좋은 후보' 채택까지 경기 수 (앞선 채택 포함 누적)
            n_cum = L.판정까지_경기.cumsum().to_numpy() if len(L) else []
            fg = next((int(n_cum[i]) for i, ch in enumerate(L.바꾼규칙) if ch in good), None) if len(L) else None
            g_, lo, hi = truth_gain(cur)
            reps_.append(dict(기간=wname, 방식=m, 채택=" → ".join(L.바꾼규칙) if len(L) else "(없음)",
                              해로운_채택=sum(ch in bad for ch in L.바꾼규칙) if len(L) else 0,
                              첫_좋은후보_채택까지_경기=fg, 채점기간_개선=f"{g_:+.2f} [{lo:+.2f}, {hi:+.2f}]"))
        cur, L = offline_hill_climb(); g_, lo, hi = truth_gain(cur)
        reps_.append(dict(기간=wname, 방식="기존 점진법 (4시즌 모아 한 번에)", 채택=" → ".join(L.바꾼규칙) if len(L) else "(없음)",
                          해로운_채택=sum(ch in bad for ch in L.바꾼규칙) if len(L) else 0,
                          첫_좋은후보_채택까지_경기=1520 if any(ch in good for ch in L.바꾼규칙) else None,
                          채점기간_개선=f"{g_:+.2f} [{lo:+.2f}, {hi:+.2f}]"))
    return pd.concat(cands), pd.DataFrame(sims), pd.DataFrame(reps_)


cands, sims, reps = replicate()
display(cands[["기간", "후보", "rps_dec", "rps_tru", "rps_tru_t", "nll_eff"]].round(2))
display(sims.set_index(["기간", "방식"]))
reps.set_index(["기간", "방식"])


### 재현 결과 (사전 실행)

**시뮬레이션 (각 500회)**

| 기간 | 좋은 후보 | 기존 점진법: 맞는 / 해로운 / 시점 | 빠른+스코어: 맞는 / 해로운 / 시점 |
|---|---|---|---|
| W1 | `k=0.04` | 60% / 6.8% / 2.0시즌 | **99% / 0.6% / 1.1시즌** |
| W2 | `k=0.04` | 49% / 0.2% / 1.0시즌 | **83% / 0.2% / 0.8시즌** |
| W3 | `k=0.04` | 66% / 2.8% / 1.0시즌 | **95% / 0.4% / 0.9시즌** |

**실제 재생 (첫 좋은 후보 채택까지 경기 수, 해로운 채택 수)**

| 기간 | 흔한 방식 | 기존 점진법 (시즌마다) | 기존 (4시즌 모아) | **빠른+스코어** |
|---|---|---|---|---|
| W1 | 380경기, 해로움 0 | 760, 0 | 1,520, 0 | **370, 0** |
| W2 | 1,140, **1** (`h=0.3`) | 1,140, 0 | 1,520, 0 | **230, 0** |
| W3 | 380, **1** (`c=0.9`) | 760, 0 | 1,520, **1** (`c=0.9`) | **280, 0** |

**채점 기간 최종 개선 (×1000, 95%):**
- W1: 빠른 방식과 기존 점진법이 같은 규칙에 도달 → 둘 다 +1.23 [−0.03, +2.57]
- W2: 빠른 +1.62 [+0.30, +2.91], 기존 +0.98 [+0.36, +1.59]
- W3: 빠른 +1.60, 기존 +1.03, 4시즌 모아 +1.71
- 모두 구간이 겹칩니다.

### 판정
1. **조건 1 (좋은 후보를 더 많이):** 세 기간 모두 통과. 83~99% 대 49~66%.
2. **조건 2 (해로운 후보를 더 적게):** W1·W3 통과, **W2는 0.2%로 같음**. 엄밀히는 2/3 통과입니다.
3. **조건 3 (실제 재생에서 해로운 채택 없음):** 세 기간 모두 통과.
4. **속도:** 첫 좋은 후보 채택은 빠른 방식이 세 기간 모두 가장 빨랐습니다(230~370경기).
   - 기존 점진법(시즌마다) 대비 W1 390경기, W2 910경기, W3 480경기 앞섰습니다.
   - 다만 W1에서 흔한 방식과의 차이는 10경기뿐이었습니다.

### 한계
- **세 기간 모두 좋은 후보가 `k=0.04`(학습 속도) 하나였습니다.** 그래서 "학습 속도를 바꾸는 개선"에서의 재현이지, 다른 종류의 개선에도 통한다는 증거는 아닙니다.
- 기간들이 서로 겹칩니다(W2 결정 = W1 채점). 독립적인 반복 세 번이 아닙니다.
- 첫 채택 뒤의 후보(예: `k=0.03`)는 시작 규칙의 이웃이 아니어서 좋고 나쁨을 분류하지 않았습니다.
- 최종 성능이 더 높다는 증거는 여전히 없습니다. 이점은 **"더 일찍, 해로운 채택 없이"**입니다.
